In [1]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path('../data')
OUTPUT_DIR = Path('../outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

inspections = pd.read_csv(DATA_DIR / 'Building_Inspections_20260519.csv', low_memory=False)
permits = pd.read_csv(DATA_DIR / 'Building_Permits_20260519.csv', low_memory=False)

print(f'Inspections: {len(inspections):,} rows x {len(inspections.columns)} cols')
print(f'Permits:     {len(permits):,} rows x {len(permits.columns)} cols')

Inspections: 686,236 rows x 33 cols
Permits:     1,288,510 rows x 53 cols


In [2]:
# Step 1: Filter to permit-type inspections only and parse date columns
insp = inspections[inspections['reference_number_type'] == 'permit'].copy()
insp['reference_number'] = insp['reference_number'].astype(str)
print(f'Permit-type inspections: {len(insp):,} of {len(inspections):,} total')

for col in ['appointment_date', 'request_taken_date', 'request_date']:
    insp[col] = pd.to_datetime(insp[col], errors='coerce')

Permit-type inspections: 606,134 of 686,236 total


/var/folders/n_/rbh3_bsn3h9cv6rx2h1bftm80000gn/T/ipykernel_31736/876996324.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  insp[col] = pd.to_datetime(insp[col], errors='coerce')
/var/folders/n_/rbh3_bsn3h9cv6rx2h1bftm80000gn/T/ipykernel_31736/876996324.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  insp[col] = pd.to_datetime(insp[col], errors='coerce')


In [3]:
# Step 3: Collapse inspections to one row per permit
def any_failed(results):
    return results.str.lower().str.contains('fail|disapprove', na=False).any()

def has_cancellation(descs):
    return descs.str.upper().str.contains('CANCELLATION|SITE VERIFICATION', na=False).any()

def has_duplicates(descs):
    # True if any inspection_description appears more than once
    return descs.duplicated().any()

insp_collapsed = insp.groupby('reference_number').agg(
    inspection_count            = ('appointment_date', 'count'),
    first_request_taken_date    = ('request_taken_date', 'min'),
    last_appointment_date       = ('appointment_date', 'max'),
    any_failed                  = ('result', any_failed),
    distinct_inspection_types   = ('inspection_description', 'nunique'),
    has_cancellation_or_siteverif = ('inspection_description', has_cancellation),
    has_duplicate_descriptions  = ('inspection_description', has_duplicates),
).reset_index()

# Truncate to date only before computing span (request_taken_date has time component)
insp_collapsed['first_request_taken_date'] = pd.to_datetime(insp_collapsed['first_request_taken_date']).dt.normalize()
insp_collapsed['last_appointment_date']    = pd.to_datetime(insp_collapsed['last_appointment_date']).dt.normalize()

insp_collapsed['span_days'] = (
    insp_collapsed['last_appointment_date'] - insp_collapsed['first_request_taken_date']
).dt.days

insp_collapsed['span_days_negative'] = insp_collapsed['span_days'] < 0

# Sequential stages: multiple distinct inspection types (not just dupes or admin rows)
insp_collapsed['has_sequential_stages'] = insp_collapsed['distinct_inspection_types'] > 1

n_neg = insp_collapsed['span_days_negative'].sum()
print(f'Collapsed to {len(insp_collapsed):,} unique permits')
print(f'Negative span_days: {n_neg:,} ({n_neg/len(insp_collapsed)*100:.1f}%) — flagged, not corrected')
print()
print('Inspection pattern flags:')
for col in ['any_failed', 'has_sequential_stages', 'has_duplicate_descriptions', 'has_cancellation_or_siteverif']:
    n = insp_collapsed[col].sum()
    print(f'  {col:<40} {n:>7,}  ({n/len(insp_collapsed)*100:.1f}%)')
insp_collapsed.head()

Collapsed to 219,200 unique permits
Negative span_days: 956 (0.4%) — flagged, not corrected

Inspection pattern flags:
  any_failed                                46,274  (21.1%)
  has_sequential_stages                    104,422  (47.6%)
  has_duplicate_descriptions                65,694  (30.0%)
  has_cancellation_or_siteverif              9,891  (4.5%)


,reference_number,inspection_count,first_request_taken_date,last_appointment_date,any_failed,distinct_inspection_types,has_cancellation_or_siteverif,has_duplicate_descriptions,span_days,span_days_negative,has_sequential_stages
0,2000011383,1,2025-11-14,2025-11-14,False,1,False,False,0,False,False
1,200002111715,1,2019-10-01,2019-10-01,False,1,False,False,0,False,False
2,200002111744,1,2018-08-17,2018-08-17,False,1,False,False,0,False,False
3,200002222456,1,2016-08-25,2016-08-25,False,1,False,False,0,False,False
4,200002262877,1,2015-08-17,2015-08-14,False,1,False,False,-3,True,False


In [4]:
# Step 3: Join collapsed inspections to permits (1-to-1)
permits = permits.rename(columns={'Permit Number': 'reference_number'})
permits['reference_number'] = permits['reference_number'].astype(str)

joined = insp_collapsed.merge(permits, on='reference_number', how='inner')
print(f'Joined: {len(joined):,} permits with both inspection history and permit metadata')

Joined: 240,862 permits with both inspection history and permit metadata


In [5]:
# Step 4: Filter to purely residential occupancy (exact R-2 or R-3 only, no mixed-use)
residential = {'R-2', 'R-3'}

def is_pure_residential(val):
    return isinstance(val, str) and val.strip() in residential

mask = (
    joined['Existing Occupancy'].apply(is_pure_residential) |
    joined['Proposed Occupancy'].apply(is_pure_residential)
)
joined = joined[mask].copy()
print(f'After residential filter: {len(joined):,} permits')

# Filter to completed permits only
joined['Completed Date'] = pd.to_datetime(joined['Completed Date'], errors='coerce')
joined = joined[joined['Completed Date'].notna()].copy()
joined['completed_year'] = joined['Completed Date'].dt.year

# Filter to 2016-2026
joined = joined[joined['completed_year'].between(2016, 2026)].copy()
joined = joined.sort_values('completed_year').reset_index(drop=True)

# Flag permits with "revision to" in Description
joined['is_revision'] = joined['Description'].str.lower().str.contains('revision to', na=False)

print(f'After completed date + year filter: {len(joined):,} permits')
print(f'Revision permits: {joined["is_revision"].sum():,} ({joined["is_revision"].mean()*100:.1f}%)')
print()
print('Permits by completed year:')
print(joined['completed_year'].value_counts().sort_index().to_string())

After residential filter: 178,884 permits
After completed date + year filter: 137,086 permits
Revision permits: 10,267 (7.5%)

Permits by completed year:
completed_year
2016    13820
2017    13678
2018    14244
2019    15139
2020    10400
2021    11539
2022    12355
2023    12738
2024    13705
2025    14410
2026     5058


In [6]:
# Work type classification based on AB 1738 categories
# Multi-label: one boolean flag per category, permits can match multiple
# Two tiers: homeowner discretion (tier 1) and inspector discretion (tier 2)

desc = joined['Description'].str.lower().fillna('')

WORK_TYPES = {
    # Tier 1: Remote at homeowner request
    'water_heater':    r'water heat',
    'hvac':            r'hvac|heat pump|furnace|boiler|air condition|mini.?split|mechanical',
    'reroof':          r're.?roof|reroofing|reroof',
    'elec_plumbing':   r'electrical|plumbing|wiring|panel|circuit|meter|gas line|gas pipe|sewer|water line',
    'solar_battery':   r'solar|photovoltaic|\bpv\b|battery storage|energy storage',
    'wildfire':        r'wildfire|fire hard|ember|defensible',
    'adu':             r'\badu\b|accessory dwelling|in-law|junior unit|\bjadu\b',
    # Tier 2: Remote at inspector discretion
    'drywall':         r'drywall|sheetrock|gypsum board',
    'siding':          r'siding|exterior finish|stucco|cladding',
    'insulation':      r'insulation|insulate',
    'windows_doors':   r'window|door replacement|door install',
    'patio_deck':      r'patio|deck|balcony|porch',
    'small_addition':  r'addition|expand|square feet|sq\.?\s?ft',
}

TIER_1 = {'water_heater', 'hvac', 'reroof', 'elec_plumbing', 'solar_battery', 'wildfire', 'adu'}
TIER_2 = {'drywall', 'siding', 'insulation', 'windows_doors', 'patio_deck', 'small_addition'}

for label, pattern in WORK_TYPES.items():
    joined[f'is_{label}'] = desc.str.contains(pattern, regex=True)

# Summary work_type string listing all matched categories (or 'other')
def work_type_summary(row):
    matched = [k for k in WORK_TYPES if row[f'is_{k}']]
    return ', '.join(matched) if matched else 'other'

joined['work_types'] = joined.apply(work_type_summary, axis=1)

tier1_cols = [f'is_{k}' for k in TIER_1]
tier2_cols = [f'is_{k}' for k in TIER_2]
joined['is_tier1'] = joined[tier1_cols].any(axis=1)
joined['is_tier2'] = joined[tier2_cols].any(axis=1)
joined['is_other'] = ~joined['is_tier1'] & ~joined['is_tier2']

print('Work type coverage:')
for label in WORK_TYPES:
    n = joined[f'is_{label}'].sum()
    print(f"  {'(T1)' if label in TIER_1 else '(T2)'} {label:<20} {n:>6,}  ({n/len(joined)*100:.1f}%)")

print()
print(f"Tier 1 (any):  {joined['is_tier1'].sum():,} ({joined['is_tier1'].mean()*100:.1f}%)")
print(f"Tier 2 (any):  {joined['is_tier2'].sum():,} ({joined['is_tier2'].mean()*100:.1f}%)")
print(f"Other:         {joined['is_other'].sum():,} ({joined['is_other'].mean()*100:.1f}%)")

Work type coverage:
  (T1) water_heater          1,278  (0.9%)
  (T1) hvac                  2,638  (1.9%)
  (T1) reroof               22,497  (16.4%)
  (T1) elec_plumbing        10,197  (7.4%)
  (T1) solar_battery           205  (0.1%)
  (T1) wildfire                256  (0.2%)
  (T1) adu                   1,225  (0.9%)
  (T2) drywall               4,103  (3.0%)
  (T2) siding                9,364  (6.8%)
  (T2) insulation            1,737  (1.3%)
  (T2) windows_doors        23,670  (17.3%)
  (T2) patio_deck           10,838  (7.9%)
  (T2) small_addition       10,000  (7.3%)

Tier 1 (any):  36,406 (26.6%)
Tier 2 (any):  46,395 (33.8%)
Other:         59,922 (43.7%)


In [7]:
# Summary table: rows = (inspection_count_group x any_failed), cols = metrics
summary_df = joined.copy()
summary_df['inspection_count_group'] = summary_df['inspection_count'].clip(upper=4).map(
    {1: '1', 2: '2', 3: '3', 4: '4+'}
)

summary = (
    summary_df.groupby(['inspection_count_group', 'any_failed'])
    .agg(
        num_permits  = ('reference_number', 'count'),
        avg_span_days= ('span_days', 'mean'),
        med_span_days= ('span_days', 'median'),
    )
    .reindex(pd.MultiIndex.from_product([['1','2','3','4+'], [False, True]], names=['inspection_count_group', 'any_failed']))
    .reset_index()
)

summary['avg_span_days'] = summary['avg_span_days'].round(1)
summary['med_span_days'] = summary['med_span_days'].round(1)
summary['num_permits']   = summary['num_permits'].map(lambda x: f'{int(x):,}' if pd.notna(x) else '-')
summary['any_failed']    = summary['any_failed'].map({False: 'No', True: 'Yes'})

summary.columns = ['Inspection Count', 'Any Failed', 'Permits', 'Avg Span Days', 'Median Span Days']
display(summary)

,Inspection Count,Any Failed,Permits,Avg Span Days,Median Span Days
0,1,No,"58,057",3.4,2.0
1,1,Yes,94,3.6,3.0
2,2,No,"25,219",116.6,63.0
3,2,Yes,"4,002",88.3,24.0
4,3,No,"14,254",162.4,99.5
5,3,Yes,"4,112",171.5,92.0
6,4+,No,"17,001",301.2,209.0
7,4+,Yes,"14,347",361.8,258.0


In [8]:
# Export full filtered dataset with all flags for Excel pivot analysis
flag_cols = (
    ['reference_number', 'completed_year', 'inspection_count', 'distinct_inspection_types',
     'span_days', 'span_days_negative', 'any_failed',
     'has_sequential_stages', 'has_duplicate_descriptions', 'has_cancellation_or_siteverif',
     'is_revision', 'work_types', 'is_tier1', 'is_tier2', 'is_other']
    + [f'is_{k}' for k in WORK_TYPES]
    + ['Description', 'Permit Type Definition', 'Existing Occupancy', 'Proposed Occupancy',
       'Estimated Cost', 'first_request_taken_date', 'last_appointment_date', 'Completed Date']
)

# Keep only columns that exist
export_cols = [c for c in flag_cols if c in joined.columns]
export_df = joined[export_cols].copy()

out_path = OUTPUT_DIR / 'building_inspections_permits_full.csv'
export_df.to_csv(out_path, index=False)
print(f'Exported {len(export_df):,} permits x {len(export_cols)} columns to {out_path}')
print()
print('Column list:')
for c in export_cols:
    print(f'  {c}')

Exported 137,086 permits x 36 columns to ../outputs/building_inspections_permits_full.csv

Column list:
  reference_number
  completed_year
  inspection_count
  distinct_inspection_types
  span_days
  span_days_negative
  any_failed
  has_sequential_stages
  has_duplicate_descriptions
  has_cancellation_or_siteverif
  is_revision
  work_types
  is_tier1
  is_tier2
  is_other
  is_water_heater
  is_hvac
  is_reroof
  is_elec_plumbing
  is_solar_battery
  is_wildfire
  is_adu
  is_drywall
  is_siding
  is_insulation
  is_windows_doors
  is_patio_deck
  is_small_addition
  Description
  Permit Type Definition
  Existing Occupancy
  Proposed Occupancy
  Estimated Cost
  first_request_taken_date
  last_appointment_date
  Completed Date


---

## San Diego County Building Inspections

> **Note:** This analysis was originally scoped to compare inspection timelines across multiple California cities. We stopped here because San Francisco was the only city where we found sufficient structured data (permit-level join with inspection history and date fields) to compute meaningful timelines. San Diego County data is explored below but lacks a "request received" timestamp, making apples-to-apples comparison with SF difficult.

Data source: [San Diego County Building Inspections](https://data.sandiegocounty.gov/Housing-and-Infrastructure/Building-Inspections/fan4-6gvy/about_data)

Key columns:
- `Request Date` — when the inspection was requested
- `Schedule Date` — when it was scheduled
- `Inspection Date` — when it actually occurred
- `Status` — Pass / Fail
- `Inspection Type` — type of inspection
- `Record Type` — permit type / work category

In [ ]:
sd = pd.read_csv(DATA_DIR / 'SanDiegoCounty_Building_Inspections_20260520.csv', low_memory=False)
print(f'Rows: {len(sd):,}  |  Columns: {len(sd.columns)}')

for col in ['Request Date', 'Schedule Date', 'Inspection Date']:
    sd[col] = pd.to_datetime(sd[col], errors='coerce')

sd['year'] = sd['Inspection Date'].dt.year

print()
print('Status values:')
print(sd['Status'].value_counts().to_string())
print()
print('Year distribution:')
print(sd['year'].value_counts().sort_index().to_string())
sd.head()

In [ ]:
city_counts = sd['City'].value_counts()
total = len(sd)
print(f'Unique cities: {city_counts.nunique()}')
print()
print(f"{'City':<30} {'Count':>8}  {'%':>6}")
print('-' * 48)
for city, n in city_counts.items():
    print(f"{str(city):<30} {n:>8,}  {n/total*100:>5.1f}%")

In [ ]:
sd = sd[sd['City'] == 'SAN DIEGO'].copy()
print(f'San Diego only: {len(sd):,} rows')